<a href="https://colab.research.google.com/github/mena-04/DoS-Stress-Testing/blob/main/testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi
!pip uninstall -y torch torchvision torchaudio vllm
!pip install -q -U uv
!uv pip install --system vllm --torch-backend=cu130
import torch
import vllm

print("vLLM:", vllm.__version__)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Fri Sep 11 10:40:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
import vllm

print("vLLM:", vllm.__version__)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

vLLM: 0.29.0
Torch: 2.13.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: Tesla T4


In [ ]:
import importlib.util

print("torchaudio installed:",
      importlib.util.find_spec("torchaudio") is not None)

torchaudio installed: False


In [ ]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [ ]:
import subprocess

log = open("/content/vllm.log", "w")

server = subprocess.Popen(
    [
        "vllm", "serve",
        "Qwen/Qwen2.5-0.5B-Instruct",
        "--dtype", "half",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.85",
        "--host", "127.0.0.1",
        "--port", "8000",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("PID:", server.pid)

PID: 9154


In [ ]:
import requests
import time

for i in range(90):

    # Has the server process already crashed?
    if server.poll() is not None:
        print("SERVER PROCESS EXITED")
        print("exit code:", server.returncode)
        break

    try:
        r = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )

        if r.status_code == 200:
            print("vLLM READY")
            print("status:", r.status_code)
            break

    except requests.RequestException:
        pass

    if i % 5 == 0:
        print("still starting...", i)

    time.sleep(2)

else:
    print("Timed out waiting for vLLM")

still starting... 0
still starting... 5
still starting... 10
still starting... 15
still starting... 20
still starting... 25
still starting... 30
still starting... 35
still starting... 40
still starting... 45
still starting... 50
still starting... 55
still starting... 60
vLLM READY
status: 200


In [ ]:
!tail -n 100 /content/vllm.log

(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347] 
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347] 
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'host': '127.0.0.1', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85}
(APIServer pid=9154) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=9154) INFO 09-11 10:53:19 [model.py:684] Resolved 

In [ ]:
import requests
import time

URL = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain AI inference in one sentence."
        }
    ],
    "max_tokens": 32,
    "temperature": 0
}

start = time.perf_counter()

r = requests.post(
    URL,
    json=payload,
    timeout=60
)

elapsed = time.perf_counter() - start

print("status:", r.status_code)
print("latency:", round(elapsed, 3), "seconds")

data = r.json()

print("response:")
print(data["choices"][0]["message"]["content"])

print("usage:")
print(data["usage"])

status: 200
latency: 0.887 seconds
response:
AI inference involves processing and analyzing large amounts of data to make predictions or decisions based on patterns and relationships within that data.
usage:
{'prompt_tokens': 37, 'total_tokens': 62, 'completion_tokens': 25, 'prompt_tokens_details': None, 'completion_tokens_details': None}


In [ ]:
m = requests.get(
    "http://127.0.0.1:8000/metrics",
    timeout=10
)

print("metrics status:", m.status_code)

wanted = [
    "vllm:num_requests_running",
    "vllm:num_requests_waiting",
    "vllm:request_success_total",
    "vllm:e2e_request_latency_seconds",
    "vllm:request_queue_time_seconds",
    "vllm:prompt_tokens_total",
    "vllm:generation_tokens_total",
]

for metric in wanted:
    print("\n---", metric, "---")
    for line in m.text.splitlines():
        if line.startswith(metric):
            print(line)

metrics status: 200

--- vllm:num_requests_running ---
vllm:num_requests_running{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0

--- vllm:num_requests_waiting ---
vllm:num_requests_waiting{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="capacity"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="deferred"} 0.0

--- vllm:request_success_total ---
vllm:request_success_total{engine="0",finished_reason="stop",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 1.0
vllm:request_success_total{engine="0",finished_reason="length",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="abort",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="error",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="repetit

# background metrics sampler

In [ ]:
import threading
import requests
import time
import re
import csv

stop_sampling = False
samples = []

def get_value(text, name):
    pattern = rf'^{re.escape(name)}\{{.*?\}}\s+([0-9.eE+-]+)'
    m = re.search(pattern, text, re.MULTILINE)
    return float(m.group(1)) if m else None

def sampler():
    while not stop_sampling:
        try:
            text = requests.get(
                "http://127.0.0.1:8000/metrics",
                timeout=2
            ).text

            samples.append({
                "timestamp": time.time(),
                "running": get_value(
                    text,
                    "vllm:num_requests_running"
                ),
                "waiting": get_value(
                    text,
                    "vllm:num_requests_waiting"
                )
            })
        except Exception:
            pass

        time.sleep(0.25)

thread = threading.Thread(target=sampler, daemon=True)
thread.start()

print("sampler started")

sampler started


# concurrent overload test

In [ ]:
import asyncio
import aiohttp
import time
import statistics

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30

async def send_one(session, i):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": EXPENSIVE_PROMPT}
        ],
        "max_tokens": 128,
        "temperature": 0
    }

    start = time.perf_counter()

    try:
        async with session.post(URL, json=payload, timeout=120) as r:
            await r.text()
            latency = time.perf_counter() - start
            return {
                "id": i,
                "status": r.status,
                "latency": latency
            }
    except Exception as e:
        return {
            "id": i,
            "status": "error",
            "latency": time.perf_counter() - start
        }

async def run_load(n=50):
    async with aiohttp.ClientSession() as session:
        tasks = [
            asyncio.create_task(send_one(session, i))
            for i in range(n)
        ]
        return await asyncio.gather(*tasks)

results = await run_load(50)

latencies = [
    r["latency"]
    for r in results
    if r["status"] == 200
]

print("completed:", len(results))
print("successful:", len(latencies))

if latencies:
    print("p50:", round(statistics.median(latencies), 3))

    sorted_lat = sorted(latencies)
    p95_index = int(0.95 * len(sorted_lat)) - 1
    print("p95:", round(sorted_lat[p95_index], 3))

    print("max:", round(max(latencies), 3))

completed: 100
successful: 100
p50: 5.603
p95: 5.7
max: 5.701


In [ ]:
stop_sampling = True
thread.join(timeout=2)

print("samples:", len(samples))
print("max running:", max(x["running"] or 0 for x in samples))
print("max waiting:", max(x["waiting"] or 0 for x in samples))

samples: 180
max running: 0
max waiting: 0
